# Pricing vanilla call options and computing implied volatilities using neural networks



**EXECUTER CETTE CELLULE PUIS REDEMARRER L'ENVIRNEMENT ET REEXECUTER** 

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy import stats
from scipy.stats import qmc
from scipy.special import erf
from scipy.special import gamma
import matplotlib.pyplot as plt


In [2]:
def cf_baseline(u, params, tau):
    """
    Characteristic function for the baseline model (Gaussian without jumps).
    
    Parameters:
    - u: Complex number or array (integration variable).
    - params: Dictionary containing model parameters:
        - 'S0': Initial asset price.
        - 'r': Risk-free rate.
        - 'sigma': Volatility.
    - tau: Time to maturity.
    
    Returns:
    - cf_value: The value of the characteristic function.
    """
    S0 = params['S0']
    r = params['r']
    sigma = params['sigma']
    
    mu = np.log(S0) + (r - 0.5 * sigma**2) * tau
    cf_value = np.exp(1j * u * mu - 0.5 * u**2 * sigma**2 * tau)
    return cf_value


In [3]:
def cf_edgeworth(u, params, tau):
    """
    Characteristic function for the Edgeworth expansion model.
    
    Parameters:
    - u: Complex number or array (integration variable).
    - params: Dictionary containing model parameters:
        - 'mu': Mean of log-returns.
        - 'sigma': Volatility.
        - 'gamma1': Skewness.
        - 'gamma2': Excess kurtosis.
    - tau: Time to maturity.
    
    Returns:
    - cf_value: The value of the characteristic function.
    """
    mu = params['mu']
    sigma = params['sigma']
    gamma1 = params['gamma1']
    gamma2 = params['gamma2']
    
    cf_value = np.exp(
        1j * u * mu - 
        0.5 * u**2 * sigma**2 + 
        (1j / 6) * u**3 * gamma1 * sigma**3 - 
        (1 / 24) * u**4 * gamma2 * sigma**4
    )
    return cf_value


In [4]:
def sinc_pricing(K, tau, r, cf, params):
    """
    Pricing European call options using the SINC method.
    
    Parameters:
    - K: Strike price (can be an array).
    - tau: Time to maturity.
    - r: Risk-free rate.
    - cf: Characteristic function to use (cf_baseline or cf_edgeworth).
    - params: Dictionary of parameters for the characteristic function.
    
    Returns:
    - option_prices: Array of option prices.
    """
    # Integration settings
    N = 128  # Number of integration points
    a = -100  # Lower limit of integration
    b = 100   # Upper limit of integration
    x = np.linspace(a, b, N)
    k = np.log(K)
    
    # Compute the integrand
    def integrand(u):
        numerator = np.exp(-1j * u * k) * cf(u - (1j * 0.5), params, tau)
        denominator = (u - (1j * 0.5)) * (u - (1j * 0.5))
        return (numerator / denominator).real
    
    # Perform the integration using the trapezoidal rule
    integrand_values = integrand(x)
    integral = np.trapz(integrand_values, x)
    
    # Compute the option price
    option_price = (np.exp(-r * tau) / np.pi) * integral
    return option_price


In [5]:
def generate_data(n_samples, model='baseline'):
    """
    Generate synthetic data for training the ANN.
    
    Parameters:
    - n_samples: Number of samples to generate.
    - model: 'baseline' or 'edgeworth' to select the characteristic function.
    
    Returns:
    - X: Input features array.
    - y: Option prices array.
    """
    # Define the parameter ranges
    S0_range = [90, 110]      # Asset price
    K_range = [80, 120]       # Strike price
    tau_range = [0.001, 0.01] # Time to maturity (ODTE)
    r_range = [0.0, 0.05]     # Risk-free rate
    sigma_range = [0.1, 0.5]  # Volatility
    gamma1_range = [-0.5, 0.5]  # Skewness (for Edgeworth)
    gamma2_range = [0.0, 1.0]   # Excess kurtosis (for Edgeworth)
    
    # LHS sampling
    sampler = qmc.LatinHypercube(d=5 if model=='baseline' else 7)
    sample = sampler.random(n_samples)
    
    # Scale the samples to the parameter ranges
    if model == 'baseline':
        S0_samples = qmc.scale(sample[:, 0], S0_range[0], S0_range[1])
        K_samples = qmc.scale(sample[:, 1], K_range[0], K_range[1])
        tau_samples = qmc.scale(sample[:, 2], tau_range[0], tau_range[1])
        r_samples = qmc.scale(sample[:, 3], r_range[0], r_range[1])
        sigma_samples = qmc.scale(sample[:, 4], sigma_range[0], sigma_range[1])
        gamma1_samples = None
        gamma2_samples = None
    else:
        S0_samples = qmc.scale(sample[:, 0], S0_range[0], S0_range[1])
        K_samples = qmc.scale(sample[:, 1], K_range[0], K_range[1])
        tau_samples = qmc.scale(sample[:, 2], tau_range[0], tau_range[1])
        r_samples = qmc.scale(sample[:, 3], r_range[0], r_range[1])
        sigma_samples = qmc.scale(sample[:, 4], sigma_range[0], sigma_range[1])
        gamma1_samples = qmc.scale(sample[:, 5], gamma1_range[0], gamma1_range[1])
        gamma2_samples = qmc.scale(sample[:, 6], gamma2_range[0], gamma2_range[1])
    
    # Compute option prices
    option_prices = []
    for i in range(n_samples):
        params = {
            'S0': S0_samples[i],
            'r': r_samples[i],
            'sigma': sigma_samples[i],
        }
        if model == 'edgeworth':
            params['mu'] = np.log(S0_samples[i])
            params['gamma1'] = gamma1_samples[i]
            params['gamma2'] = gamma2_samples[i]
            cf = cf_edgeworth
        else:
            cf = cf_baseline
        price = sinc_pricing(
            K=np.array([K_samples[i]]),
            tau=tau_samples[i],
            r=r_samples[i],
            cf=cf,
            params=params
        )
        option_prices.append(price)
    
    # Prepare input features
    if model == 'baseline':
        X = np.column_stack((S0_samples, K_samples, tau_samples, r_samples, sigma_samples))
    else:
        X = np.column_stack((S0_samples, K_samples, tau_samples, r_samples, sigma_samples, gamma1_samples, gamma2_samples))
    y = np.array(option_prices)
    return X, y


In [6]:
class OptionPricingANN(nn.Module):
    def __init__(self, input_dim):
        """
        Neural network model for option pricing.
        
        Parameters:
        - input_dim: Number of input features.
        """
        super(OptionPricingANN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 1)
        
    def forward(self, x):
        """
        Forward pass of the neural network.
        """
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x.flatten()


In [7]:
def train_model(model, optimizer, criterion, dataloader, epochs=10):
    """
    Train the neural network model.
    
    Parameters:
    - model: The neural network model.
    - optimizer: Optimizer for updating the weights.
    - criterion: Loss function.
    - dataloader: DataLoader for the training data.
    - epochs: Number of training epochs.
    """
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
        avg_loss = total_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")


In [8]:
# Generate data
n_samples = 10000
X, y = generate_data(n_samples, model='baseline')  # Change to 'edgeworth' for the second specification

# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X)
y_tensor = torch.FloatTensor(y)

# Create dataset and dataloader
dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

# Initialize the model
input_dim = X.shape[1]
model = OptionPricingANN(input_dim)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Train the model
train_model(model, optimizer, criterion, dataloader, epochs=20)


ValueError: Sample is not a 2D array